## Day 2 - Part 5: 전이학습과 파인튜닝: 거인의 어깨 위에서 시작하기

### 개요

Day 2의 앞선 파트들에서 우리는 DNN의 구조를 설계하고(`Part 1`), 하이퍼파라미터를 튜닝했으며(`Part 2`), 모델을 평가하고(`Part 3`), 마침내 이미지를 '보는' CNN(`Part 4`)까지 직접 만들어 보았습니다. 

하지만 CNN 모델을 '처음부터(from scratch)' 학습시키는 것은 엄청난 양의 데이터와 시간, 그리고 컴퓨팅 자원을 필요로 하는 힘든 여정입니다.

여기서 우리는 이런 질문을 던지게 됩니다. 

"이미 세상의 똑똑한 사람들이 수백만 장의 이미지로 잘 훈련시켜 놓은 강력한 모델이 있다면, 굳이 우리가 바퀴를 다시 발명해야 할까?"

이번 파트에서는 이 질문에 대한 가장 현명한 해답인 `전이학습(Transfer Learning)` 과 `미세조정(Fine-Tuning)` 에 대해 깊이 있게 탐구합니다. 

이는 마치 세계 최고의 요리사가 수년간 쌓아온 비법 레시피(사전 학습된 모델)를 전수받아, 나의 상황에 맞게 살짝 변형하여(미세조정) 빠르고 쉽게 최고의 요리를 만들어내는 것과 같습니다.  

우리는 '거인의 어깨' 위에 올라서서, 더 적은 데이터와 시간으로도 훨씬 뛰어난 성능을 내는 모델을 만드는 강력한 기술을 배우게 될 것입니다.

`이번 파트의 학습 목표:`

* 전이학습(Transfer Learning)의 개념과 그 필요성을 이해할 수 있습니다. 

* 수백만 장의 데이터로 학습된 `사전 학습 모델(Pre-trained Model)` 을 불러오고 활용할 수 있습니다. 
* 전이학습의 두 가지 핵심 전략인 `특징 추출(Feature Extraction)` 과 `미세조정(Fine-Tuning)` 의 차이를 이해하고 설명할 수 있습니다. 
* PyTorch에서 모델의 특정 층을 `동결(Freeze)` 시키고(`requires_grad=False`), 원하는 층만 학습시키는 방법을 구현할 수 있습니다. 
* `torchvision.models`를 사용하여 원하는 사전 학습 모델을 불러오고, 새로운 과제에 맞게 출력층을 수정할 수 있습니다. 
* `개 vs 고양이` 데이터셋을 사용하여, 전이학습을 통해 이미지 분류 모델을 실제로 구축하고 성능을 극대화하는 과정을 경험합니다. 


### 1. 왜 '처음부터' 학습하지 않을까? (전이학습의 필요성)

우리가 Part 4에서 CNN을 만들었듯, 신경망은 처음에는 무작위 가중치로 시작하여 수많은 데이터를 보며 점차 똑똑해집니다. 

하지만 매우 깊은 신경망이 이미지의 복잡하고 미세한 패턴(예: 동물의 털 질감, 미묘한 빛의 반사)까지 학습하려면, ImageNet과 같이 100만 장이 넘는 방대한 데이터셋이 필요합니다.  

현실의 많은 문제에서는 이 정도 규모의 데이터를 구하기가 거의 불가능합니다.

`사전 학습 모델(Pre-trained Model)` 은 바로 이 문제를 해결해 줍니다. 

Google, Meta 같은 연구기관들이 ImageNet과 같은 대규모 데이터셋으로 몇 주, 몇 달간 학습시켜 놓은 모델(예: ResNet, VGG, MobileNet 등)을 우리는 단 몇 줄의 코드로 가져와 사용할 수 있습니다. 

이 모델들은 이미 이미지의 보편적인 특징(선, 면, 질감, 형태 등)을 학습하는 방법을 알고 있습니다. 우리는 이 지식을 '전이'받아 우리의 특정 문제에 적용하기만 하면 됩니다.

`전이학습의 핵심 장점:`

* `데이터 부족 문제 해결`: 내가 가진 데이터가 수백, 수천 장에 불과하더라도, 사전 학습 모델이 가진 풍부한 지식을 활용하여 높은 일반화 성능을 얻을 수 있습니다. 

* `학습 시간 및 비용 단축`: 바닥부터 시작하는 것이 아니므로, 모델이 훨씬 빠르게 수렴하고 학습에 필요한 시간과 자원이 극적으로 줄어듭니다. 
* `더 높은 성능 달성`: 직접 만든 어설픈 모델보다, 방대한 데이터로 검증된 전문가 모델을 기반으로 하므로 더 높은 최종 성능을 기대할 수 있습니다. 


### 2. 전이학습의 두 가지 핵심 전략

전이학습은 크게 두 가지 접근 방식으로 나뉩니다. "사전 학습된 모델의 지식을 얼마나 신뢰하고, 얼마나 바꿀 것인가?"에 대한 선택의 문제입니다.

* `특징 추출 (Feature Extraction)`: 사전 학습된 모델을 '만능 특징 추출기'로 사용하는 방식입니다. 모델의 대부분을 그대로 두고, 마지막 출력 부분만 우리 문제에 맞게 새로 만들어 학습시킵니다. 

* `미세조정 (Fine-Tuning)`: 사전 학습된 모델의 지식을 출발점으로 삼되, 우리 데이터에 더 잘 맞도록 모델의 일부를 '재교육'하는 방식입니다. 

<br>

| 전략 | 특징 추출 (Feature Extraction) | 미세조정 (Fine-Tuning) |
| :--- | :--- | :--- |
| `개념` | 합성곱 기반층(몸통)은 `동결(Freeze)` 하고,<br>분류기(머리)만 새로 학습 | 합성곱 기반층의 `일부도 함께` 학습하여<br>가중치를 새로운 데이터에 맞게 조정 |
| `비유` | 전문가용 카메라의 렌즈는 그대로 쓰고,<br>사진을 인화하는 방식만 바꿈 | 전문가용 카메라 렌즈의<br>초점과 조리개를 살짝 재조정함 |
| `언제?` | 데이터가 `매우 적을 때`. 과적합 방지에 유리. | 데이터가 `어느 정도 충분할 때`. 더 높은 성능 기대. |
| `장점` | 학습이 매우 빠르고 안정적임.  | 모델을 새로운 데이터에 최적화 가능.  |
| `단점` | 기존 모델의 특징에 전적으로 의존.  | 학습이 더 오래 걸리고, 학습률 설정에 민감함.  |



#### 2.1. 코드 실습: 사전 학습 모델 불러오고 구조 변경하기

먼저 어떤 전략을 쓰든 공통적으로 필요한 '사전 학습 모델 로드'와 '출력층 교체'를 실습해 보겠습니다. 

PyTorch의 `torchvision.models`는 다양한 사전 학습 모델을 제공합니다. 여기서는 가볍고 성능이 좋은 `ResNet-18`을 사용해 보겠습니다.

In [3]:
import torch
import torch.nn as nn
import torchvision.models as models

# ImageNet으로 사전 학습된 ResNet-18 모델을 불러옵니다.
# pretrained=True 옵션이 가중치를 함께 다운로드합니다.
model_path = "../models/torch/resnet18.pth"
model = models.resnet18(pretrained=True)

# 모델의 구조를 출력해봅니다.
# print(model)

# 모델의 마지막 부분을 보면 'fc'라는 완전 연결층이 있습니다.
# (fc): Linear(in_features=512, out_features=1000, bias=True)
# out_features=1000은 ImageNet의 클래스가 1000개이기 때문입니다.
print("기존 출력층:", model.fc)

기존 출력층: Linear(in_features=512, out_features=1000, bias=True)


/Users/dante/workspace/dante-code/class/star_track_python/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/dante/workspace/dante-code/class/star_track_python/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
# 모델의 전체 구조를 출력합니다
print("=== ResNet-18 모델 구조 ===")
print(model)

=== ResNet-18 모델 구조 ===
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu

In [6]:
import torch.onnx # ONNX로 모델 저장
import netron # Netron으로 시각화

# 더미 입력 데이터 생성 (배치 크기 1, 3채널, 224x224 이미지)
dummy_input = torch.randn(1, 3, 224, 224)

# ONNX 파일로 저장
onnx_path = "../models/torch/resnet18_transfer_learning.onnx"
torch.onnx.export(
    model,                     # 변환할 모델
    dummy_input,              # 모델 입력 예시
    onnx_path,                # 저장할 파일 경로
    export_params=True,       # 모델 가중치 포함
    opset_version=11,         # ONNX 버전
    do_constant_folding=True, # 최적화
    input_names=['input'],    # 입력 이름
    output_names=['output'],  # 출력 이름
    dynamic_axes={            # 동적 배치 크기 지원
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

netron.start(onnx_path)

모델이 ../models/torch/resnet18_transfer_learning.onnx에 저장되었습니다.


In [9]:
model.fc.in_features

512

In [10]:
# 우리는 '개'와 '고양이', 2개의 클래스만 분류하면 되므로,
# 이 출력층을 새로운 것으로 교체해야 합니다.

# 기존 fc층의 입력 피처 수를 가져옵니다.
num_features = model.fc.in_features

# 새로운 출력층을 정의합니다. (입력: 512, 출력: 2)
model.fc = nn.Linear(num_features, 2)

print("새로운 출력층:", model.fc)

새로운 출력층: Linear(in_features=512, out_features=2, bias=True)


이제 이 모델을 가지고 '특징 추출'과 '미세조정'을 구현하는 방법을 알아봅시다. 

핵심은 `requires_grad` 속성을 제어하여 특정 층의 가중치가 업데이트(학습)되지 않도록 '동결(Freeze)'시키는 것입니다.



#### 2.2. 코드 실습: 모델 층 동결 및 해제하기

`전략 1: 특징 추출을 위한 동결`

특징 추출에서는 새로 교체한 `fc` 층을 제외한 모든 층의 가중치를 동결합니다.

In [11]:

# 1. 먼저 모델의 모든 파라미터를 동결합니다.
for param in model.parameters():
    param.requires_grad = False

# 2. 새로 교체한 fc 층의 파라미터만 동결을 해제하여 학습 대상으로 설정합니다.
for param in model.fc.parameters():
    param.requires_grad = True

# 이제 옵티마이저는 requires_grad=True인 파라미터만 업데이트합니다.
# model.fc.parameters()만 전달해도 되고, 아래처럼 필터링해서 전달해도 됩니다.
params_to_update = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params_to_update, lr=0.001)

print("학습할 파라미터 수:", len(params_to_update))

학습할 파라미터 수: 2


`전략 2: 미세조정을 위한 부분 해제`

미세조정에서는 `fc` 층과 더불어, 사전 학습된 층 중 일부(보통 끝부분)의 동결을 추가로 해제합니다. ResNet의 경우 `layer4` 블록을 함께 학습시켜 보겠습니다.

In [12]:
# (특징 추출 학습이 끝난 모델을 가정)
# 추가로 layer4의 동결을 해제합니다.
for param in model.layer4.parameters():
    param.requires_grad = True

# 학습할 파라미터를 다시 필터링합니다.
params_to_update_finetune = [p for p in model.parameters() if p.requires_grad]

# 옵티마이저를 새로 정의합니다.
# 중요: 미세조정 시에는 매우 작은 학습률을 사용해야 합니다!
# 기존의 학습된 가중치를 파괴하지 않기 위함입니다.
optimizer_finetune = torch.optim.Adam(params_to_update_finetune, lr=0.0001)

print("학습할 파라미터 그룹 수 (fc, layer4):", len(params_to_update_finetune))

학습할 파라미터 그룹 수 (fc, layer4): 17


### 3. 종합 실습: 전이학습으로 개와 고양이 분류하기

이제 배운 모든 것을 종합하여 Kaggle의 `Dogs vs. Cats` 데이터셋을 분류하는 전체 과정을 실습해 보겠습니다. (편의상 전체 데이터 중 일부 샘플을 사용한다고 가정합니다.)

#### 3.1. 데이터 준비: Kaggle Dogs vs. Cats

가장 먼저 데이터를 불러오고 모델에 맞는 형태로 변환해야 합니다. `torchvision.transforms`를 사용하여 이미지의 크기를 조정하고, 텐서로 변환하며, 정규화를 수행합니다.

`매우 중요`: 사전 학습 모델을 사용할 때는, 해당 모델이 학습될 때와 `동일한 방식으로 정규화`를 해주어야 합니다. 

대부분의 `torchvision` 모델은 ImageNet 데이터로 학습되었으며, 평균 `[0.485, 0.456, 0.406]`, 표준편차 `[0.229, 0.224, 0.225]`를 사용합니다. 

In [13]:
import kaggle
kaggle.api.authenticate()
kaggle.api.dataset_download_files("tongpython/cat-and-dog", path="../datasets/dl/dogs-vs-cats", unzip=True)

Dataset URL: https://www.kaggle.com/datasets/tongpython/cat-and-dog


In [14]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# 데이터가 저장된 경로
# training_set/training_set/
#   ├── cats/
#   │   ├── cat.0.jpg
#   │   └── ...
#   └── dogs/
#       ├── dog.0.jpg
#       └── ...
# test_set/test_set/
#   ├── cats/
#   │   ├── cat.0.jpg
#   │   └── ...
#   └── dogs/
#       ├── dog.0.jpg
#       └── ...

data_dir = '../datasets/dl/dogs-vs-cats'

# 데이터 변환(전처리) 파이프라인 정의
# 훈련 데이터에는 약간의 변형(Data Augmentation)을 주어 과적합을 방지합니다.
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 검증(테스트) 데이터는 변형 없이 크기 조정과 정규화만 수행합니다.
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ImageFolder를 사용하여 폴더 구조로부터 데이터셋을 생성합니다.
train_dataset = ImageFolder(data_dir + '/training_set/training_set', transform=train_transform)
val_dataset = ImageFolder(data_dir + '/test_set/test_set', transform=val_transform)

# DataLoader를 생성합니다.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class_names = train_dataset.classes
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### 3.2. 전략 1: 특징 추출(Feature Extraction) 모델 학습

이제 본격적으로 모델을 학습시켜 봅시다. 먼저 특징 추출 방식으로, 새로 추가한 `fc` 층만 빠르게 학습시킵니다.

In [15]:
# 1. 모델 불러오고 출력층 교체
model_fe = models.resnet18(pretrained=True)
num_ftrs = model_fe.fc.in_features
model_fe.fc = nn.Linear(num_ftrs, len(class_names))

# 2. 모든 층 동결 후 fc층만 해제
for param in model_fe.parameters():
    param.requires_grad = False
for param in model_fe.fc.parameters():
    param.requires_grad = True

model_fe = model_fe.to(device)

# 3. 손실 함수와 옵티마이저 정의
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_fe.fc.parameters(), lr=0.001)

# --- 학습 루프 (생략, Day2-Part1 코드와 유사) ---
# for epoch in range(num_epochs):
#     # 훈련 (model_fe.train())
#     # 검증 (model_fe.eval())
# -------------------------------------------------

# 아마도 단 5~10 에포크만으로도 90% 이상의 높은 검증 정확도를 달성할 것입니다.
# 이것이 바로 전이학습의 위력입니다! 

/Users/dante/workspace/dante-code/class/star_track_python/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/dante/workspace/dante-code/class/star_track_python/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


#### 3.3. 전략 2: 미세조정(Fine-Tuning)으로 성능 끌어올리기

특징 추출 학습으로 이미 좋은 성능을 얻은 모델을 기반으로, `layer4`를 추가로 학습시켜 성능을 한계까지 끌어올려 보겠습니다.

In [17]:
import copy
# (앞서 학습시킨 model_fe를 그대로 사용)

# 1. layer4의 동결을 해제
for param in model_fe.layer4.parameters():
    param.requires_grad = True

# 2. 미세조정을 위한 옵티마이저 새로 정의 (매우 작은 학습률이 핵심!)
params_to_update_ft = [p for p in model_fe.parameters() if p.requires_grad]
optimizer_ft = torch.optim.Adam(params_to_update_ft, lr=0.0001) # 1e-4

# 3. 미세조정 학습 루프
num_epochs_finetune = 10
best_acc = 0.0

for epoch in range(num_epochs_finetune):
    print(f'Epoch {epoch+1}/{num_epochs_finetune}')
    print('-' * 10)
    
    # 훈련 단계
    model_fe.train()
    running_loss = 0.0
    running_corrects = 0
    
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer_ft.zero_grad()
        
        with torch.set_grad_enabled(True):
            outputs = model_fe(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer_ft.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
    
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = running_corrects.double() / len(train_loader.dataset)
    
    print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
    
    # 검증 단계
    model_fe.eval()
    running_loss = 0.0
    running_corrects = 0
    
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        with torch.no_grad():
            outputs = model_fe(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
    
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_acc = running_corrects.double() / len(val_loader.dataset)
    
    print(f'Val Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
    
    # 최고 성능 모델 저장
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        best_model = copy.deepcopy(model_fe.state_dict())
    
    print()

print(f'Best val Acc: {best_acc:.4f}')

# 최고 성능 모델로 복원
model_fe.load_state_dict(best_model)

# 검증 정확도가 1~2% 라도 더 상승한다면 미세조정은 성공입니다.

Epoch 1/10
----------
Train Loss: 0.1262 Acc: 0.9483
Val Loss: 0.0661 Acc: 0.9748

Epoch 2/10
----------
Train Loss: 0.1059 Acc: 0.9542
Val Loss: 0.0465 Acc: 0.9852

Epoch 3/10
----------
Train Loss: 0.0958 Acc: 0.9608
Val Loss: 0.0447 Acc: 0.9857

Epoch 4/10
----------
Train Loss: 0.0959 Acc: 0.9580
Val Loss: 0.0412 Acc: 0.9847

Epoch 5/10
----------
Train Loss: 0.0831 Acc: 0.9643
Val Loss: 0.0538 Acc: 0.9827

Epoch 6/10
----------
Train Loss: 0.0878 Acc: 0.9619
Val Loss: 0.0460 Acc: 0.9842

Epoch 7/10
----------
Train Loss: 0.0838 Acc: 0.9650
Val Loss: 0.0383 Acc: 0.9876

Epoch 8/10
----------
Train Loss: 0.0799 Acc: 0.9646
Val Loss: 0.0508 Acc: 0.9847

Epoch 9/10
----------
Train Loss: 0.0810 Acc: 0.9671
Val Loss: 0.0453 Acc: 0.9852

Epoch 10/10
----------
Train Loss: 0.0787 Acc: 0.9668
Val Loss: 0.0398 Acc: 0.9886

Best val Acc: 0.9886


<All keys matched successfully>

#### 3.4. 최종 모델 평가 및 예측 시각화

학습이 완료된 최종 모델을 사용하여, 테스트 이미지에 대한 예측을 수행하고 그 결과를 눈으로 직접 확인해 봅시다.

In [19]:
import plotly.express as px
import numpy as np
from PIL import Image

# 1. 임의의 테스트 이미지 열기
test_image_path = '../datasets/dl/dogs-vs-cats/some_cat.jpeg' # 테스트할 이미지 경로
image = Image.open(test_image_path)

# 2. 이미지를 모델 입력에 맞게 변환 (val_transform 사용)
# unsqueeze(0)를 통해 배치 차원 [1, 3, 224, 224]를 만들어 줍니다.
image_tensor = val_transform(image).unsqueeze(0).to(device)

# 3. 모델로 예측하기
model_fe.eval() # 반드시 평가 모드로 설정!
with torch.no_grad():
    outputs = model_fe(image_tensor)
    _, predicted_idx = torch.max(outputs, 1)
    predicted_label = class_names[predicted_idx.item()]

# 4. 결과 시각화
# Plotly로 시각화하기 위해 텐서를 Numpy 배열로 변환
# (C, H, W) -> (H, W, C) 순서로 축 변경 필요
img_to_show = image_tensor.squeeze().cpu().numpy()
# 정규화된 이미지를 원래대로 되돌리는 과정 (시각화를 위해)
img_to_show = img_to_show * np.array([0.229, 0.224, 0.225])[:, None, None] + np.array([0.485, 0.456, 0.406])[:, None, None]
img_to_show = np.clip(img_to_show, 0, 1) # 값 범위를 [0, 1]로 유지

fig = px.imshow(np.transpose(img_to_show, (1, 2, 0)),
                title=f"Model's Prediction: {predicted_label}")
fig.show()